# EasyPush image-conedir qualification (vanilla image CRL, binary NCE)

One-seed qualification of **vanilla contrastive RL on pixels**:
`fetch_push_image_conedir` = the state-qualified conedir task with observations
replaced by two flattened 64x64x3 uint8 renders (frame + goal frame) from the
original FetchPushImage's fixed `camera2`. **Task dynamics unchanged; no causal
changes.** Relabel goal = the full future frame (identity obs_to_goal, as in
the original image envs).

Pipeline: pinned install -> clone+checkout tested commit -> mount Drive ->
**environment/render audit (stops on failure)** -> one-seed **100k**
qualification with TensorBoard + init/early/mid/final/best/latest checkpoints +
resume-from-latest -> greedy vs random-floor vs scripted-oracle eval +
cone-bin success + rollout GIF -> **PASS / WARN / FAIL** summary.

Leave `SMOKE = True` for a fast first pass, then set it `False` for the real run.


In [ ]:
# 1. Dependencies WITHOUT disturbing Colab's preinstalled GPU JAX.
#
# Colab ships a GPU-matched jax/jaxlib/numpy (jax 0.7.x). Installing
# dm-haiku/optax the normal way drags in jax-ecosystem deps that mismatch the
# native jaxlib -> the kernel hard-crashes on the next GPU op (repeated 'kernel
# restarted', no traceback). PROVEN-SAFE recipe (validated end-to-end on Colab,
# incl. egl rendering): install the jax-adjacent libs with --no-deps, then add
# their small pure-python deps + the env stack, HOLDING jax/jaxlib/numpy at
# Colab's versions. Run on a FRESH runtime (Runtime > Disconnect and delete).
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # do not grab all VRAM
os.environ["MUJOCO_GL"] = "egl"                         # headless render backend
import jax, jaxlib, numpy, subprocess, sys
hold = [f"jax=={jax.__version__}", f"jaxlib=={jaxlib.__version__}",
        f"numpy=={numpy.__version__}"]
def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
print("Colab JAX", jax.__version__, "| devices:", jax.devices())
pip("--no-deps", "dm-haiku", "optax", "chex")            # jax libs: no dep changes
pip("jmp", "tabulate", "toolz", "etils", "tensorboardX",  # pure-python deps + TB
    "gymnasium-robotics", "mujoco", "imageio-ffmpeg", *hold)
print("post-install JAX", jax.__version__, "| devices:", jax.devices())
# If devices() shows CPU only, switch to a GPU runtime
# (Runtime > Change runtime type > T4 GPU) and re-run.


In [ ]:
# 2. Clone the fork and checkout the tested commit.
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
COMMIT = "0c14fbf"   # tested commit for this notebook (image-obs conedir)
!git fetch -q origin && git checkout -q $COMMIT
!git log -1 --oneline
assert os.path.exists("scripts/smoke_image_conedir.py"), \
    "checkout is missing the image-conedir smoke -- wrong commit/fork?"


In [ ]:
# 3. Mount Google Drive (all checkpoints + artifacts saved here).
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# 4. Parameters + guard-railed binary-NCE config (image obs).
import os, json
import numpy as np
from crl.config import Config
from crl.train import train

ENV    = 'fetch_push_image_conedir'
SEED   = 0
RESUME = False          # True to continue from latest.pkl
SMOKE  = True           # True = fast integration smoke; set False for the real run

if SMOKE:
    STEPS, MINREP, RANDS, EVAL, EVALEP, LOG = 3_000, 1_000, 1_000, 1_500, 2, 1_000
else:
    STEPS, MINREP, RANDS, EVAL, EVALEP, LOG = 100_000, 10_000, 10_000, 10_000, 10, 5_000

BASE    = '/content/drive/MyDrive/easypush_image_qual'
RUN_DIR = f'{BASE}/{ENV}_s{SEED}' + ('_smoke' if SMOKE else '')
os.makedirs(RUN_DIR, exist_ok=True)

def build_config():
    cfg = Config(
        env_name=ENV, use_td=False, twin_q=False,        # binary NCE
        random_goals=0.5, entropy_coefficient=0.0,
        max_number_of_steps=STEPS,
        # uint8 frames: buffer sized to the run (100k steps ~ 2.4 GB RAM;
        # the 1M default would be ~24 GB and OOM the Colab VM).
        max_replay_size=STEPS,
        min_replay_size=MINREP, random_steps=RANDS,
        num_sgd_steps_per_step=4, batch_size=256,
        eval_every_steps=EVAL, eval_episodes=EVALEP, log_every_steps=LOG,
        seed=SEED, ckpt_dir=RUN_DIR, resume=RESUME, tensorboard=True,
        guard_abort=True,
    )
    assert cfg.use_td is False and cfg.twin_q is False, 'must be binary NCE'
    assert cfg.max_number_of_steps <= 200_000, 'qualification run, not 1M'
    return cfg

print(f'ENV={ENV} SEED={SEED} SMOKE={SMOKE} STEPS={STEPS} RESUME={RESUME}')
print('RUN_DIR:', RUN_DIR)


In [ ]:
# 5. Environment/render audit -- STOP before training if it fails.
#    Gates: wiring / goal-leak (0 pixels may change when u.goal moves) /
#    goal-image content / scripted-oracle ceiling / one finite NCE update
#    through the uint8+conv path (also proves EGL + GPU conv work here).
!python -m scripts.smoke_image_conedir
audit = json.load(open('artifacts/push_image_probe/smoke_image_conedir.json'))
from IPython.display import Image, display
for f in ('pair_reset.png', 'pair_goal.png', 'mid_push_frame.png'):
    display(Image(f'artifacts/push_image_probe/{f}', width=192))
assert audit['verdict'] == 'PASS', \
    'image-conedir environment audit FAILED -- stopping before training.'
print('image env audit: PASS  (oracle=%.2f random=%.2f)' % (
    audit['gate4_oracle']['oracle_success'],
    audit['gate4_oracle']['random_success']))


In [ ]:
# 6. Launch TensorBoard (critic/actor loss, logits gap, cat-acc, success,
#    pixel-space final/min distance).
tb_logdir = RUN_DIR + '/tb'
%load_ext tensorboard
%tensorboard --logdir $tb_logdir


In [ ]:
# 7. Train (guard-railed). Milestones init/early/mid/final/best/latest are
#    saved to Drive; RESUME=True continues from latest.pkl. NaN guard in-loop.
cfg = build_config()
print('OK:', ENV, 'binary NCE on pixels, seed', SEED, '->', RUN_DIR)
state = train(cfg)


In [ ]:
# 8. Report: greedy vs random floor vs scripted oracle, cone-bin success,
#    rollout GIF, PASS/WARN/FAIL verdict.
import numpy as np, jax, jax.numpy as jnp, imageio, json
from crl import envs as envs_mod, networks as networks_mod
from crl import checkpoint as ckpt_mod
from crl.config import Config
from crl.report_push import _oracle_action

IMG = 64 * 64 * 3
rcfg = Config(env_name=ENV)
env = envs_mod.make_env(ENV, rcfg, seed=123)
u = env._env.unwrapped
nets = networks_mod.make_networks(
    obs_dim=rcfg.obs_dim, goal_dim=rcfg.goal_dim, action_dim=rcfg.action_dim,
    repr_dim=int(rcfg.repr_dim), repr_norm=rcfg.repr_norm,
    repr_norm_temp=rcfg.repr_norm_temp,
    hidden_layer_sizes=rcfg.hidden_layer_sizes, twin_q=rcfg.twin_q,
    use_image_obs=True)
step, tstate = ckpt_mod.load_checkpoint(f'{RUN_DIR}/best.pkl')

@jax.jit
def _greedy(o):
    return nets.sample_eval(nets.policy_network.apply(tstate.policy_params, o), None)

def rollout(kind, n, gif_eps=0):
    rng = np.random.default_rng(0)
    succ, bins, frames = [], [], []
    for k in range(n):
        obs = env.reset()
        theta = float(np.arctan2(env._last_dir[1], env._last_dir[0]))
        hit = 0.0
        for _ in range(env.max_episode_steps):
            if kind == 'greedy':
                a = np.asarray(_greedy(jnp.asarray(obs[None]))[0])
            elif kind == 'random':
                a = rng.uniform(-1, 1, 4).astype(np.float32)
            else:                                     # oracle: reads sim state
                o = u._get_obs()
                a = _oracle_action(np.concatenate(
                    [o['observation'], o['desired_goal']]).astype(np.float32))
            obs, r, _, _ = env.step(a)
            hit = max(hit, float(r))
            if k < gif_eps:
                frames.append(np.kron(obs[:IMG].reshape(64, 64, 3),
                                      np.ones((4, 4, 1))).astype(np.uint8))
        succ.append(hit)
        bins.append((abs(np.degrees(theta)), hit))
    return float(np.mean(succ)), bins, frames

EPS = 8 if SMOKE else 50
greedy_s, bins, frames = rollout('greedy', EPS, gif_eps=2)
rand_s, _, _ = rollout('random', max(4, EPS // 2))
orac_s, _, _ = rollout('oracle', max(4, EPS // 2))

gif = f'{RUN_DIR}/rollout.gif'
imageio.mimsave(gif, frames, duration=0.08)
from IPython.display import Image as IPImage, display
display(IPImage(gif, width=256))

by_bin = {}
for lo, hi in ((0, 20), (20, 40), (40, 60)):
    xs = [h for d, h in bins if lo <= d < hi]
    by_bin[f'|theta| {lo}-{hi}deg'] = (
        (round(float(np.mean(xs)), 3), len(xs)) if xs else (None, 0))

mh = json.load(open(f'{RUN_DIR}/metrics.json'))
hist = mh['history'] if isinstance(mh, dict) and 'history' in mh else mh
last = [r for r in hist if 'critic_loss' in r][-1:]
cat_acc = last[0].get('categorical_accuracy') if last else None

issues = []
if orac_s < 0.9:
    issues.append(('WARN', f'oracle ceiling {orac_s:.2f} < 0.9 -- env/render health?'))
if greedy_s >= max(0.85, rand_s + 0.2):
    verdict = 'PASS'
elif greedy_s >= rand_s + 0.1:
    verdict = 'WARN'
    issues.append(('WARN', f'greedy {greedy_s:.2f} above the random floor '
                           f'{rand_s:.2f} but below the 0.85 bar'))
else:
    verdict = 'FAIL'
    issues.append(('FAIL', f'greedy {greedy_s:.2f} not above random floor {rand_s:.2f}'))
if SMOKE:
    issues.append(('WARN', 'SMOKE run: verdict is integration-only, not task-level'))

summary = {'verdict': verdict, 'greedy_success': greedy_s,
           'random_floor': rand_s, 'oracle_ceiling': orac_s,
           'success_by_cone_bin': by_bin, 'cat_acc_last': cat_acc,
           'ckpt_step': int(step), 'episodes': EPS, 'smoke': SMOKE}
with open(f'{RUN_DIR}/qualification.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 56)
print('IMAGE-CONEDIR QUALIFICATION:', verdict)
print('=' * 56)
for sev, msg in issues:
    print(f'  [{sev}] {msg}')
print(json.dumps(summary, indent=2))
